In [1]:
# Standard library
import os
import sys
import glob
import shutil
from pathlib import Path

# Data handling
import pandas as pd

# PySpark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType,
    BooleanType,
)

### Project configuration

In [4]:
PROJECT_ROOT = Path(
    r"D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics"
).resolve()
# Where process_raw_data.py already wrote the six flat CSVs
RAW_CSV_DIR = Path(os.environ.get("RAW_CSV_DIR", PROJECT_ROOT / "data" / "raw_data_csv"))

# Where this notebook writes its cleaned outputs
CLEANED_DIR = Path(os.environ.get("CLEANED_DIR", PROJECT_ROOT / "data" / "cleaned"))

for d in (CLEANED_DIR,):
    d.mkdir(parents=True, exist_ok=True)

EXPECTED_CSV_FILES = {
    "stops":              "timetable_stops.csv",
    "vehicle_journeys":   "timetable_vehicle_journeys.csv",
    "stop_times":         "timetable_stop_times.csv",
    "disruptions":        "disruptions.csv",
    "fares":              "fares.csv",
    "location":           "location_pings.csv",
}

print(f"Project root:  {PROJECT_ROOT}")
print(f"Raw CSV dir:   {RAW_CSV_DIR}")
print(f"Cleaned dir:   {CLEANED_DIR}")

missing = [f for f in EXPECTED_CSV_FILES.values() if not (RAW_CSV_DIR / f).exists()]
assert not missing, (
    f"Missing raw CSV file(s): {missing}. Run process_raw_data.py first, or set "
    f"RAW_CSV_DIR to point at the folder that has these files."
)
print("\nAll six expected raw CSV files are present.")

Project root:  D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics
Raw CSV dir:   D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\data\raw_data_csv
Cleaned dir:   D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\data\cleaned

All six expected raw CSV files are present.


### Spark session

In [6]:
N_CORES = int(os.environ.get("SPARK_CORES", "8"))

spark = (
    SparkSession.builder
    .appName("BusRoute_01_DataCollectionCleaning")
    .master(f"local[{N_CORES}]")
    .config("spark.sql.shuffle.partitions", str(N_CORES * 2))
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "4g"))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version:       {spark.version}")
print(f"Cores configured:    {N_CORES}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Shuffle partitions:  {spark.conf.get('spark.sql.shuffle.partitions')}")
print("\nSpark UI available at:", spark.sparkContext.uiWebUrl)

Spark version:       3.5.8
Cores configured:    8
Default parallelism: 8
Shuffle partitions:  16

Spark UI available at: http://DESKTOP-P7PE4MO:4040


## Reusable cleaning & validation helpers

In [7]:
def read_csv_validated(path: Path, schema: StructType, label: str):
    '''Read a CSV against an explicit schema, verifying the header matches
    and surfacing any row Spark could not parse as `_corrupt_record`.'''
    actual_header = pd.read_csv(path, nrows=0).columns.tolist()
    expected_cols = [f.name for f in schema.fields]
    if actual_header != expected_cols:
        raise ValueError(
            f"[{label}] header mismatch.\n"
            f"  expected: {expected_cols}\n"
            f"  actual:   {actual_header}\n"
            f"Update the schema definition to match the real file before proceeding."
        )

    schema_with_corrupt = StructType(
        schema.fields + [StructField("_corrupt_record", StringType(), True)]
    )
    df = (
        spark.read
        .option("header", True)
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .schema(schema_with_corrupt)
        .csv(str(path))
    )
    return df


def report_and_drop_corrupt(df, label: str):
    '''Print how many rows failed to parse against the schema, then drop the
    corrupt-record tracking column (its job is done once we've reported it).'''
    n_corrupt = df.filter(F.col("_corrupt_record").isNotNull()).count()
    if n_corrupt:
        print(f"  ! {label}: {n_corrupt:,} row(s) failed schema parsing and will be dropped.")
    else:
        print(f"  {label}: 0 rows failed schema parsing.")
    return df.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")

In [8]:
def standardize_column_names(df):
    '''Lowercase, strip, and underscore-ify every column name.'''
    for c in df.columns:
        clean = c.strip().lower().replace(" ", "_")
        if clean != c:
            df = df.withColumnRenamed(c, clean)
    return df


def trim_string_columns(df):
    '''Trim whitespace on every string column and convert empty-after-trim
    strings into real nulls so missing-value checks catch them.'''
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    for c in string_cols:
        df = df.withColumn(c, F.trim(F.col(c)))
        df = df.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))
    return df

In [9]:
def profile(df, key_cols, label=""):
    '''Row count + per-column null counts + duplicate count on `key_cols`,
    computed with the caller's cached DataFrame (no extra reads triggered here
    beyond what the caller already cached).'''
    n = df.count()

    null_report = df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns
    ]).collect()[0].asDict()

    n_distinct_keys = df.dropDuplicates(key_cols).count()
    n_dupes = n - n_distinct_keys

    print(f"\n--- Profile: {label} ({n:,} rows) ---")
    any_missing = False
    for col_name, missing_count in null_report.items():
        if missing_count > 0:
            any_missing = True
            pct = 100 * missing_count / n if n else 0
            print(f"  null  {col_name:<30}: {missing_count:>10,} ({pct:.1f}%)")
    if not any_missing:
        print("  No missing values found.")
    print(f"  duplicate rows on {key_cols}: {n_dupes:,} ({n:,} -> {n_distinct_keys:,})")

    return {"rows": n, "nulls": null_report, "duplicates": n_dupes}


class DQTracker:
    '''Collects before/after row counts for the final data-quality summary table.'''

    def __init__(self, name):
        self.name = name
        self.original_rows = None
        self.after_dedup_rows = None
        self.after_missing_rows = None
        self.final_rows = None

    def as_row(self):
        dup = (self.original_rows - self.after_dedup_rows
               if None not in (self.original_rows, self.after_dedup_rows) else 0)
        miss = (self.after_dedup_rows - self.after_missing_rows
                if None not in (self.after_dedup_rows, self.after_missing_rows) else 0)
        inval = (self.after_missing_rows - self.final_rows
                 if None not in (self.after_missing_rows, self.final_rows) else 0)
        return {
            "Dataset": self.name,
            "Original Rows": self.original_rows,
            "Final Rows": self.final_rows,
            "Duplicates Removed": dup,
            "Missing/Invalid Handled": miss,
            "Invalid Removed": inval,
        }


dq_trackers = []
print("Helper functions ready.")

Helper functions ready.


In [10]:
def save_clean(df, name: str, n_rows_hint=None):
    parquet_path = CLEANED_DIR / f"{name}.parquet"
    df.write.mode("overwrite").parquet(str(parquet_path))

    csv_folder = CLEANED_DIR / f"_{name}_csv_tmp"
    df.coalesce(1).write.mode("overwrite").option("header", True).csv(str(csv_folder))
    part_file = glob.glob(str(csv_folder / "part-*.csv"))[0]
    final_csv = CLEANED_DIR / f"{name}.csv"
    if final_csv.exists():
        final_csv.unlink()
    shutil.move(part_file, str(final_csv))
    shutil.rmtree(csv_folder)

    row_note = f"{n_rows_hint:,} rows" if n_rows_hint is not None else ""
    print(f"Saved -> {parquet_path} and {final_csv} ({row_note})")

## Per-dataset ingestion, cleaning, and validation

### timetable_stops

In [11]:
stops_schema = StructType([
    StructField("source_file", StringType(), True),
    StructField("stop_point_ref", StringType(), True),
    StructField("common_name", StringType(), True),
])

stops_raw = read_csv_validated(RAW_CSV_DIR / EXPECTED_CSV_FILES["stops"], stops_schema, "timetable_stops")
stops_raw = standardize_column_names(stops_raw)
stops_raw = report_and_drop_corrupt(stops_raw, "timetable_stops")
stops_raw = trim_string_columns(stops_raw).cache()

tracker_stops = DQTracker("timetable_stops")
tracker_stops.original_rows = stops_raw.count()
print(f"Loaded {tracker_stops.original_rows:,} rows")
stops_raw.printSchema()
stops_raw.show(5, truncate=False)

  timetable_stops: 0 rows failed schema parsing.
Loaded 20,570 rows
root
 |-- source_file: string (nullable = true)
 |-- stop_point_ref: string (nullable = true)
 |-- common_name: string (nullable = true)

+-----------------------------------------------------------------------------------------------------------+--------------+----------------------------+
|source_file                                                                                                |stop_point_ref|common_name                 |
+-----------------------------------------------------------------------------------------------------------+--------------+----------------------------+
|1-None--SCCM-CACA-2025-06-01-Live_CB_2025_06_01_-_School_back_+_C__SCCM_PF0000459_27_20250619-BODS_V1_1.xml|0500CCITY242  |Arbury Campkin Rd           |
|1-None--SCCM-CACA-2025-06-01-Live_CB_2025_06_01_-_School_back_+_C__SCCM_PF0000459_27_20250619-BODS_V1_1.xml|0500CCITY334  |Kings Hedges Hawkins Road   |
|1-None--SCCM-CACA-2025-

In [12]:
_ = profile(stops_raw, ["stop_point_ref"], "timetable_stops (raw)")
n_null_ref = stops_raw.filter(F.col("stop_point_ref").isNull()).count()
print(f"Invalid records -- null stop_point_ref: {n_null_ref:,}")


--- Profile: timetable_stops (raw) (20,570 rows) ---
  No missing values found.
  duplicate rows on ['stop_point_ref']: 11,675 (20,570 -> 8,895)
Invalid records -- null stop_point_ref: 0


In [13]:
stops_clean = stops_raw.dropDuplicates(["stop_point_ref"])
tracker_stops.after_dedup_rows = stops_clean.count()

stops_clean = stops_clean.filter(F.col("stop_point_ref").isNotNull())
stops_clean = stops_clean.fillna({"common_name": "UNKNOWN_STOP_NAME"})
tracker_stops.after_missing_rows = stops_clean.count()

# No numeric-range validation applies to this dataset -- cardinality is the
# relevant check here (each stop_point_ref should now be unique).
tracker_stops.final_rows = tracker_stops.after_missing_rows
stops_clean = stops_clean.cache()

n_unique = stops_clean.select("stop_point_ref").distinct().count()
assert n_unique == tracker_stops.final_rows, "stop_point_ref is not unique after cleaning"
print(f"Rows after cleaning: {tracker_stops.final_rows:,} (unique stop_point_ref confirmed)")

dq_trackers.append(tracker_stops)
save_clean(stops_clean, "timetable_stops", tracker_stops.final_rows)

Rows after cleaning: 8,895 (unique stop_point_ref confirmed)
Saved -> D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\data\cleaned\timetable_stops.parquet and D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\data\cleaned\timetable_stops.csv (8,895 rows)


### timetable_vehicle_journeys

In [14]:
vj_schema = StructType([
    StructField("source_file", StringType(), True),
    StructField("vehicle_journey_code", StringType(), True),
    StructField("service_ref", StringType(), True),
    StructField("line_ref", StringType(), True),
    StructField("line_name", StringType(), True),
    StructField("operator_ref", StringType(), True),
    StructField("journey_pattern_ref", StringType(), True),
    StructField("scheduled_departure_time", StringType(), True),
])

vj_raw = read_csv_validated(RAW_CSV_DIR / EXPECTED_CSV_FILES["vehicle_journeys"], vj_schema, "vehicle_journeys")
vj_raw = standardize_column_names(vj_raw)
vj_raw = report_and_drop_corrupt(vj_raw, "vehicle_journeys")
vj_raw = trim_string_columns(vj_raw).cache()

tracker_vj = DQTracker("timetable_vehicle_journeys")
tracker_vj.original_rows = vj_raw.count()
print(f"Loaded {tracker_vj.original_rows:,} rows")
vj_raw.printSchema()
vj_raw.show(5, truncate=False)

  vehicle_journeys: 0 rows failed schema parsing.
Loaded 35,997 rows
root
 |-- source_file: string (nullable = true)
 |-- vehicle_journey_code: string (nullable = true)
 |-- service_ref: string (nullable = true)
 |-- line_ref: string (nullable = true)
 |-- line_name: string (nullable = true)
 |-- operator_ref: string (nullable = true)
 |-- journey_pattern_ref: string (nullable = true)
 |-- scheduled_departure_time: string (nullable = true)

+-----------------------------------------------------------------------------------------------------------+--------------------+------------+-------------------+---------+------------+-------------------+------------------------+
|source_file                                                                                                |vehicle_journey_code|service_ref |line_ref           |line_name|operator_ref|journey_pattern_ref|scheduled_departure_time|
+------------------------------------------------------------------------------------------

In [15]:
_ = profile(vj_raw, ["vehicle_journey_code", "source_file"], "vehicle_journeys (raw)")

bad_time = ~F.col("scheduled_departure_time").rlike(r"^\d{1,2}:\d{2}:\d{2}$")
n_bad_time = vj_raw.filter(F.col("scheduled_departure_time").isNotNull() & bad_time).count()
print(f"Invalid records -- malformed scheduled_departure_time pattern: {n_bad_time:,}")


--- Profile: vehicle_journeys (raw) (35,997 rows) ---
  No missing values found.
  duplicate rows on ['vehicle_journey_code', 'source_file']: 0 (35,997 -> 35,997)
Invalid records -- malformed scheduled_departure_time pattern: 0


In [16]:
vj_clean = vj_raw.dropDuplicates(["vehicle_journey_code", "source_file"])
tracker_vj.after_dedup_rows = vj_clean.count()

vj_clean = vj_clean.filter(
    F.col("vehicle_journey_code").isNotNull() &
    F.col("line_ref").isNotNull() &
    F.col("scheduled_departure_time").isNotNull()
)
vj_clean = vj_clean.fillna({"line_name": "UNKNOWN_LINE_NAME"})
tracker_vj.after_missing_rows = vj_clean.count()

# scheduled_departure_ts anchors the clock time to a fixed synthetic date
# (1970-01-01) purely so we can do time-of-day arithmetic (hour, duration).
# It must NOT be used to derive calendar-dependent features like day-of-week
# or month -- the source data has no calendar date, only a clock time, so any
# such feature would be constant. See the pipeline review notes.
vj_clean = vj_clean.withColumn(
    "scheduled_departure_ts",
    F.to_timestamp(F.concat(F.lit("1970-01-01 "), F.col("scheduled_departure_time")), "yyyy-MM-dd HH:mm:ss")
)
vj_clean = vj_clean.filter(F.col("scheduled_departure_ts").isNotNull()).cache()
tracker_vj.final_rows = vj_clean.count()
print(f"Rows after cleaning: {tracker_vj.final_rows:,}")

_ = profile(vj_clean, ["vehicle_journey_code", "source_file"], "vehicle_journeys (cleaned)")
print("\nJourneys per line (top 10):")
vj_clean.groupBy("line_ref").count().orderBy(F.desc("count")).show(10)

dq_trackers.append(tracker_vj)
save_clean(vj_clean, "timetable_vehicle_journeys", tracker_vj.final_rows)

Rows after cleaning: 35,997

--- Profile: vehicle_journeys (cleaned) (35,997 rows) ---
  No missing values found.
  duplicate rows on ['vehicle_journey_code', 'source_file']: 0 (35,997 -> 35,997)

Journeys per line (top 10):
+--------------------+-----+
|            line_ref|count|
+--------------------+-----+
| SCOX:PH0005863:12:1| 1352|
|SCOX:PH0005863:11...| 1118|
| SCOX:PH0005863:43:8|  852|
| SCOX:PH0005863:13:2|  846|
| SCGL:PH0005031:10:1|  750|
|SCOX:PH0005863:13:2A|  710|
|SCGL:PH0005031:13:10|  692|
|SCGL:PH0005031:182:C|  668|
|SCGL:PH0005031:180:A|  664|
|SCOX:PH0005863:16:S1|  618|
+--------------------+-----+
only showing top 10 rows

Saved -> D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\data\cleaned\timetable_vehicle_journeys.parquet and D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\data\cleaned\timetable_vehicle_journeys.csv (35,997 rows)


### timetable_stop_times

In [17]:
stop_times_schema = StructType([
    StructField("source_file", StringType(), True),
    StructField("vehicle_journey_code", StringType(), True),
    StructField("line_ref", StringType(), True),
    StructField("stop_point_ref", StringType(), True),
    StructField("stop_sequence", IntegerType(), True),
    StructField("scheduled_time", StringType(), True),
])

st_raw = read_csv_validated(RAW_CSV_DIR / EXPECTED_CSV_FILES["stop_times"], stop_times_schema, "stop_times")
st_raw = standardize_column_names(st_raw)
st_raw = report_and_drop_corrupt(st_raw, "stop_times")
st_raw = trim_string_columns(st_raw)
st_raw = st_raw.repartition(N_CORES * 4, "line_ref").cache()

tracker_st = DQTracker("timetable_stop_times")
tracker_st.original_rows = st_raw.count()
print(f"Loaded {tracker_st.original_rows:,} rows, {st_raw.rdd.getNumPartitions()} partitions")
st_raw.printSchema()
st_raw.show(5, truncate=False)

  stop_times: 0 rows failed schema parsing.
Loaded 1,126,331 rows, 32 partitions
root
 |-- source_file: string (nullable = true)
 |-- vehicle_journey_code: string (nullable = true)
 |-- line_ref: string (nullable = true)
 |-- stop_point_ref: string (nullable = true)
 |-- stop_sequence: integer (nullable = true)
 |-- scheduled_time: string (nullable = true)

+------------------------------------------------------------------------------------------------------+--------------------+---------------------+--------------+-------------+--------------+
|source_file                                                                                           |vehicle_journey_code|line_ref             |stop_point_ref|stop_sequence|scheduled_time|
+------------------------------------------------------------------------------------------------------+--------------------+---------------------+--------------+-------------+--------------+
|9_BD-None--SCCM-EABE-2025-06-01-LIVE_BD_2025_06_01_PAY_FILE__SC

In [18]:
_ = profile(st_raw, ["vehicle_journey_code", "stop_point_ref", "stop_sequence"], "stop_times (raw)")
n_negative_seq = st_raw.filter(F.col("stop_sequence") < 0).count()
print(f"Invalid records -- negative stop_sequence: {n_negative_seq:,}")


--- Profile: stop_times (raw) (1,126,331 rows) ---
  No missing values found.
  duplicate rows on ['vehicle_journey_code', 'stop_point_ref', 'stop_sequence']: 199,850 (1,126,331 -> 926,481)
Invalid records -- negative stop_sequence: 0


In [19]:
st_clean = st_raw.dropDuplicates(["vehicle_journey_code", "stop_point_ref", "stop_sequence"])
tracker_st.after_dedup_rows = st_clean.count()

st_clean = st_clean.filter(
    F.col("vehicle_journey_code").isNotNull() &
    F.col("stop_point_ref").isNotNull() &
    F.col("scheduled_time").isNotNull() &
    (F.col("stop_sequence") >= 0)
)
tracker_st.after_missing_rows = st_clean.count()

# scheduled_ts: same synthetic-date anchoring caveat as vehicle_journeys above.
st_clean = st_clean.withColumn(
    "scheduled_ts",
    F.to_timestamp(F.concat(F.lit("1970-01-01 "), F.col("scheduled_time")), "yyyy-MM-dd HH:mm:ss")
)
st_clean = st_clean.filter(F.col("scheduled_ts").isNotNull())
st_clean = st_clean.repartition(N_CORES * 4, "line_ref").cache()
tracker_st.final_rows = st_clean.count()
print(f"Rows after cleaning: {tracker_st.final_rows:,}")

_ = profile(st_clean, ["vehicle_journey_code", "stop_point_ref", "stop_sequence"], "stop_times (cleaned)")
print("\nSummary statistics -- stop_sequence (route length indicator):")
st_clean.describe("stop_sequence").show()

dq_trackers.append(tracker_st)
save_clean(st_clean, "timetable_stop_times", tracker_st.final_rows)

Rows after cleaning: 926,481

--- Profile: stop_times (cleaned) (926,481 rows) ---
  No missing values found.
  duplicate rows on ['vehicle_journey_code', 'stop_point_ref', 'stop_sequence']: 0 (926,481 -> 926,481)

Summary statistics -- stop_sequence (route length indicator):
+-------+------------------+
|summary|     stop_sequence|
+-------+------------------+
|  count|            926481|
|   mean|  19.9051076060923|
| stddev|15.558389601912447|
|    min|                 0|
|    max|                94|
+-------+------------------+

Saved -> D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\data\cleaned\timetable_stop_times.parquet and D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\data\cleaned\timetable_stop_times.csv (926,481 rows)


### location_pings

In [20]:
location_schema = StructType([
    StructField("polled_at_utc", StringType(), True),
    StructField("recorded_at_time", StringType(), True),
    StructField("valid_until_time", StringType(), True),
    StructField("item_identifier", StringType(), True),
    StructField("vehicle_ref", StringType(), True),
    StructField("line_ref", StringType(), True),
    StructField("published_line_name", StringType(), True),
    StructField("operator_ref", StringType(), True),
    StructField("direction_ref", StringType(), True),
    StructField("data_frame_ref", StringType(), True),
    StructField("dated_vehicle_journey_ref", StringType(), True),
    StructField("origin_ref", StringType(), True),
    StructField("origin_name", StringType(), True),
    StructField("destination_ref", StringType(), True),
    StructField("destination_name", StringType(), True),
    StructField("origin_aimed_departure_time", StringType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("bearing", DoubleType(), True),
    StructField("block_ref", StringType(), True),
    StructField("vehicle_journey_ref", StringType(), True),
])

location_raw = read_csv_validated(RAW_CSV_DIR / EXPECTED_CSV_FILES["location"], location_schema, "location_pings")
location_raw = standardize_column_names(location_raw)
location_raw = report_and_drop_corrupt(location_raw, "location_pings")
location_raw = trim_string_columns(location_raw).cache()

tracker_loc = DQTracker("location_pings")
tracker_loc.original_rows = location_raw.count()
print(f"Loaded {tracker_loc.original_rows:,} rows")
location_raw.printSchema()
location_raw.show(5, truncate=False)

  location_pings: 0 rows failed schema parsing.
Loaded 21,048 rows
root
 |-- polled_at_utc: string (nullable = true)
 |-- recorded_at_time: string (nullable = true)
 |-- valid_until_time: string (nullable = true)
 |-- item_identifier: string (nullable = true)
 |-- vehicle_ref: string (nullable = true)
 |-- line_ref: string (nullable = true)
 |-- published_line_name: string (nullable = true)
 |-- operator_ref: string (nullable = true)
 |-- direction_ref: string (nullable = true)
 |-- data_frame_ref: string (nullable = true)
 |-- dated_vehicle_journey_ref: string (nullable = true)
 |-- origin_ref: string (nullable = true)
 |-- origin_name: string (nullable = true)
 |-- destination_ref: string (nullable = true)
 |-- destination_name: string (nullable = true)
 |-- origin_aimed_departure_time: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- bearing: double (nullable = true)
 |-- block_ref: string (nullable = true)
 |-- vehicle_j

In [21]:
_ = profile(location_raw, ["item_identifier"], "location_pings (raw)")

n_bad_lat = location_raw.filter(~F.col("latitude").between(-90, 90)).count()
n_bad_lon = location_raw.filter(~F.col("longitude").between(-180, 180)).count()
n_bad_bearing = location_raw.filter(
    F.col("bearing").isNotNull() & ~F.col("bearing").between(0, 359)
).count()
print(f"Invalid records -- latitude out of range:  {n_bad_lat:,}")
print(f"Invalid records -- longitude out of range: {n_bad_lon:,}")
print(f"Invalid records -- bearing out of range:   {n_bad_bearing:,}")


--- Profile: location_pings (raw) (21,048 rows) ---
  null  destination_name              :        125 (0.6%)
  null  origin_aimed_departure_time   :        344 (1.6%)
  null  bearing                       :      4,063 (19.3%)
  null  block_ref                     :      7,854 (37.3%)
  null  vehicle_journey_ref           :     20,334 (96.6%)
  duplicate rows on ['item_identifier']: 10,521 (21,048 -> 10,527)
Invalid records -- latitude out of range:  0
Invalid records -- longitude out of range: 0
Invalid records -- bearing out of range:   0


In [22]:
loc_clean = location_raw.dropDuplicates(["item_identifier"])
tracker_loc.after_dedup_rows = loc_clean.count()

loc_clean = loc_clean.filter(
    F.col("recorded_at_time").isNotNull() &
    F.col("latitude").isNotNull() &
    F.col("longitude").isNotNull() &
    F.col("line_ref").isNotNull() &
    F.col("latitude").between(-90, 90) &
    F.col("longitude").between(-180, 180)
)
tracker_loc.after_missing_rows = loc_clean.count()

loc_clean = loc_clean.withColumn(
    "recorded_ts", F.try_to_timestamp("recorded_at_time")
).withColumn(
    "origin_aimed_ts", F.try_to_timestamp("origin_aimed_departure_time")
).withColumn(
    "bearing", F.when(F.col("bearing").between(0, 359), F.col("bearing")).otherwise(None)
)
tracker_loc.final_rows = loc_clean.count()
loc_clean = loc_clean.cache()
print(f"Rows after cleaning: {tracker_loc.final_rows:,}")

_ = profile(loc_clean, ["item_identifier"], "location_pings (cleaned)")
print("\nSummary statistics -- latitude / longitude / bearing:")
loc_clean.describe(["latitude", "longitude", "bearing"]).show()

dq_trackers.append(tracker_loc)
save_clean(loc_clean, "location_pings", tracker_loc.final_rows)

Rows after cleaning: 10,527

--- Profile: location_pings (cleaned) (10,527 rows) ---
  null  destination_name              :         25 (0.2%)
  null  origin_aimed_departure_time   :         91 (0.9%)
  null  bearing                       :      2,432 (23.1%)
  null  block_ref                     :        235 (2.2%)
  null  vehicle_journey_ref           :     10,411 (98.9%)
  null  origin_aimed_ts               :         91 (0.9%)
  duplicate rows on ['item_identifier']: 0 (10,527 -> 10,527)

Summary statistics -- latitude / longitude / bearing:
+-------+-------------------+-------------------+------------------+
|summary|           latitude|          longitude|           bearing|
+-------+-------------------+-------------------+------------------+
|  count|              10527|              10527|              8095|
|   mean|  51.72851271273867| -1.260954379025364| 190.1229153798641|
| stddev|0.08253548493439342|0.12902241722679655|102.66380337417577|
|    min|          51.550175|     

### disruptions

In [23]:
disruptions_schema = StructType([
    StructField("source_file", StringType(), True),
    StructField("situation_number", StringType(), True),
    StructField("creation_time", StringType(), True),
    StructField("participant_ref", StringType(), True),
    StructField("version", StringType(), True),
    StructField("progress", StringType(), True),
    StructField("misc_reason", StringType(), True),
    StructField("planned", StringType(), True),
    StructField("validity_start", StringType(), True),
    StructField("summary", StringType(), True),
    StructField("description", StringType(), True),
])

dis_raw = read_csv_validated(RAW_CSV_DIR / EXPECTED_CSV_FILES["disruptions"], disruptions_schema, "disruptions")
dis_raw = standardize_column_names(dis_raw)
dis_raw = report_and_drop_corrupt(dis_raw, "disruptions")
dis_raw = trim_string_columns(dis_raw).cache()

tracker_dis = DQTracker("disruptions")
tracker_dis.original_rows = dis_raw.count()
print(f"Loaded {tracker_dis.original_rows:,} rows")
dis_raw.printSchema()
dis_raw.show(5, truncate=False)

  disruptions: 0 rows failed schema parsing.
Loaded 450 rows
root
 |-- source_file: string (nullable = true)
 |-- situation_number: string (nullable = true)
 |-- creation_time: string (nullable = true)
 |-- participant_ref: string (nullable = true)
 |-- version: string (nullable = true)
 |-- progress: string (nullable = true)
 |-- misc_reason: string (nullable = true)
 |-- planned: string (nullable = true)
 |-- validity_start: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- description: string (nullable = true)

+-----------+------------------------------------+------------------------+---------------+-------+--------+-----------+-------+--------------+--------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [24]:
_ = profile(dis_raw, ["situation_number", "version"], "disruptions (raw)")
print("\nDistinct 'progress' values before standardisation (case check):")
dis_raw.select("progress").distinct().show()


--- Profile: disruptions (raw) (450 rows) ---
  null  misc_reason                   :        109 (24.2%)
  null  validity_start                :        449 (99.8%)
  duplicate rows on ['situation_number', 'version']: 0 (450 -> 450)

Distinct 'progress' values before standardisation (case check):
+--------+
|progress|
+--------+
|    open|
|     109|
+--------+



In [25]:
dis_clean = dis_raw.dropDuplicates(["situation_number", "version"])
tracker_dis.after_dedup_rows = dis_clean.count()

dis_clean = dis_clean.filter(
    F.col("situation_number").isNotNull() & F.col("creation_time").isNotNull()
)
dis_clean = dis_clean.fillna({"misc_reason": "unspecified"})
tracker_dis.after_missing_rows = dis_clean.count()

dis_clean = dis_clean.withColumn(
    "creation_ts", F.try_to_timestamp("creation_time")
).withColumn(
    "validity_start_ts", F.try_to_timestamp("validity_start")
).withColumn(
    "progress", F.lower(F.col("progress"))
).withColumn(
    "planned", (F.lower(F.col("planned")) == "true").cast(BooleanType())
).withColumn(
    "version", F.col("version").cast(IntegerType())
)
tracker_dis.final_rows = dis_clean.count()
dis_clean = dis_clean.cache()
print(f"Rows after cleaning: {tracker_dis.final_rows:,}")

_ = profile(dis_clean, ["situation_number", "version"], "disruptions (cleaned)")
print("\nDisruption reason breakdown:")
dis_clean.groupBy("misc_reason").count().orderBy(F.desc("count")).show()

dq_trackers.append(tracker_dis)
save_clean(dis_clean, "disruptions", tracker_dis.final_rows)

Rows after cleaning: 450

--- Profile: disruptions (cleaned) (450 rows) ---
  null  validity_start                :        449 (99.8%)
  null  creation_ts                   :          1 (0.2%)
  null  validity_start_ts             :        450 (100.0%)
  duplicate rows on ['situation_number', 'version']: 0 (450 -> 450)

Disruption reason breakdown:
+------------------+-----+
|       misc_reason|count|
+------------------+-----+
|         roadworks|  218|
|       unspecified|  109|
|        roadClosed|   52|
|insufficientDemand|   26|
|      specialEvent|   20|
|    routeDiversion|    8|
|          incident|    7|
|           unknown|    6|
|        congestion|    2|
|               208|    1|
|         vandalism|    1|
+------------------+-----+

Saved -> D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\data\cleaned\disruptions.parquet and D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\data\cleaned\disruptions.csv 

### fares

In [26]:
fares_schema = StructType([
    StructField("source_file", StringType(), True),
    StructField("publication_timestamp", StringType(), True),
    StructField("participant_ref", StringType(), True),
    StructField("operator_ref", StringType(), True),
    StructField("line_ref", StringType(), True),
    StructField("from_date", StringType(), True),
    StructField("description", StringType(), True),
])

fares_raw = read_csv_validated(RAW_CSV_DIR / EXPECTED_CSV_FILES["fares"], fares_schema, "fares")
fares_raw = standardize_column_names(fares_raw)
fares_raw = report_and_drop_corrupt(fares_raw, "fares")
fares_raw = trim_string_columns(fares_raw).cache()

tracker_fares = DQTracker("fares")
tracker_fares.original_rows = fares_raw.count()
print(f"Loaded {tracker_fares.original_rows:,} rows")
fares_raw.printSchema()
fares_raw.show(5, truncate=False)

  fares: 0 rows failed schema parsing.
Loaded 3,435 rows
root
 |-- source_file: string (nullable = true)
 |-- publication_timestamp: string (nullable = true)
 |-- participant_ref: string (nullable = true)
 |-- operator_ref: string (nullable = true)
 |-- line_ref: string (nullable = true)
 |-- from_date: string (nullable = true)
 |-- description: string (nullable = true)

+--------------------------------------------------------------------------------------+----------------------------+---------------+------------+--------------------+--------------------+--------------------------------------+
|source_file                                                                           |publication_timestamp       |participant_ref|operator_ref|line_ref            |from_date           |description                           |
+--------------------------------------------------------------------------------------+----------------------------+---------------+------------+--------------------+---

In [27]:
_ = profile(fares_raw, ["participant_ref", "line_ref", "from_date"], "fares (raw)")


--- Profile: fares (raw) (3,435 rows) ---
  null  line_ref                      :        422 (12.3%)
  duplicate rows on ['participant_ref', 'line_ref', 'from_date']: 3,307 (3,435 -> 128)


In [28]:
fares_clean = fares_raw.dropDuplicates(["participant_ref", "line_ref", "from_date"])
tracker_fares.after_dedup_rows = fares_clean.count()

fares_clean = fares_clean.filter(
    F.col("participant_ref").isNotNull() & F.col("line_ref").isNotNull()
)
fares_clean = fares_clean.fillna({"description": "No description provided"})
tracker_fares.after_missing_rows = fares_clean.count()

fares_clean = fares_clean.withColumn(
    "from_date_ts", F.try_to_timestamp("from_date")
).withColumn(
    "publication_ts", F.try_to_timestamp("publication_timestamp")
)
tracker_fares.final_rows = fares_clean.count()
fares_clean = fares_clean.cache()
print(f"Rows after cleaning: {tracker_fares.final_rows:,}")

_ = profile(fares_clean, ["participant_ref", "line_ref", "from_date"], "fares (cleaned)")
print("\nFare publications per operator:")
fares_clean.groupBy("operator_ref").count().orderBy(F.desc("count")).show()

dq_trackers.append(tracker_fares)
save_clean(fares_clean, "fares", tracker_fares.final_rows)

Rows after cleaning: 126

--- Profile: fares (cleaned) (126 rows) ---
  No missing values found.
  duplicate rows on ['participant_ref', 'line_ref', 'from_date']: 0 (126 -> 126)

Fare publications per operator:
+------------+-----+
|operator_ref|count|
+------------+-----+
|    noc:SCGL|   84|
|    noc:SCOX|   42|
+------------+-----+

Saved -> D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\data\cleaned\fares.parquet and D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\data\cleaned\fares.csv (126 rows)


In [29]:
summary_df = pd.DataFrame([t.as_row() for t in dq_trackers])
summary_df["% Retained"] = (100 * summary_df["Final Rows"] / summary_df["Original Rows"]).round(1)
print(summary_df.to_string(index=False))

assert all(t.final_rows and t.final_rows > 0 for t in dq_trackers), \
    "One or more datasets ended up with zero rows after cleaning -- investigate before continuing."
print("\nAll six datasets cleaned and saved.")

                   Dataset  Original Rows  Final Rows  Duplicates Removed  Missing/Invalid Handled  Invalid Removed  % Retained
           timetable_stops          20570        8895               11675                        0                0        43.2
timetable_vehicle_journeys          35997       35997                   0                        0                0       100.0
      timetable_stop_times        1126331      926481              199850                        0                0        82.3
            location_pings          21048       10527               10521                        0                0        50.0
               disruptions            450         450                   0                        0                0       100.0
                     fares           3435         126                3307                        2                0         3.7

All six datasets cleaned and saved.


In [30]:
spark.stop()
print("Spark session stopped. Notebook 01 complete.")

Spark session stopped. Notebook 01 complete.
